In [1]:
import h5py
import json

import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge, LinearRegression, SGDRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF
import time
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import train_test_split

In [2]:
with h5py.File('solutions.h5', 'r') as f:
    # List all groups/datasets
    dataset = f["data"]   # This does NOT load data
    dataset = dataset[:]
    print(dataset.shape)        # Fast
    coef = f["coeffs"]
    coef = coef[:]
    print(coef.shape)

(8000, 64, 128, 1)
(8000, 129)


# Framework 1

In [6]:
coef_fr1 = coef[::20]
# get training and test datasets
num0 = int(0.8 * coef_fr1.shape[0])
num0_test = coef_fr1.shape[0] - num0

X_train = coef_fr1[:num0]
X_test = coef_fr1[num0:]


num = int(0.8 * dataset.shape[0])
Y_train = dataset[:num,:,:,0].reshape(num0,20*64*128)
Y_test = dataset[num:,:,:,0]

print('Train shapes', X_train.shape, Y_train.shape) 
print('Test shapes', X_test.shape, Y_test.shape)

## out-of-distribution test
#X_ood = coef_ood[::20]
#Y_ood = dataset_ood[:,:,:,0]
#print(X_ood.shape, Y_ood.shape)

Train shapes (320, 129) (320, 163840)
Test shapes (80, 129) (1600, 64, 128)


In [11]:
##################### RBF kernel
bandwidth = 0.1
kernel = RBF(length_scale = bandwidth)
model = GaussianProcessRegressor(kernel, alpha = 1e-10, optimizer=None)

model.fit(X_train, Y_train)

pred = model.predict(X_test)
pred = pred.reshape(Y_test.shape)

# reshape prediction and Y_test
e = np.mean(np.linalg.norm(pred - Y_test, axis = (1,2))/np.linalg.norm(Y_test, axis = (1,2)))
print(f'Gaussian kernel: test error is {e:.2e}.\n')    

##### Out-of-distributon
#pred = model.predict(X_ood)
#pred = pred.reshape(Y_ood.shape)
#e = np.mean(np.linalg.norm(pred - Y_ood, axis = (1,2))/np.linalg.norm(Y_ood, axis = (1,2)))
#print(f'Gaussian kernel: test error is {e:.2e}.\n')    

Gaussian kernel: test error is 1.84e-03.



In [15]:
##################### Matern kernel
bandwidth = 0.1
nu = 5/2
kernel = Matern(length_scale = bandwidth, nu=nu)
model = GaussianProcessRegressor(kernel, alpha = 1e-10, optimizer=None)

model.fit(X_train, Y_train)

pred = model.predict(X_test)
pred = pred.reshape(Y_test.shape)

# reshape prediction and Y_test
e = np.mean(np.linalg.norm(pred - Y_test, axis = (1,2))/np.linalg.norm(Y_test, axis = (1,2)))
print(f'Matern kernel: test error is {e:.2e}.\n')    

##### Out-of-distributon
#pred = model.predict(X_ood)
#pred = pred.reshape(Y_ood.shape)
#e = np.mean(np.linalg.norm(pred - Y_ood, axis = (1,2))/np.linalg.norm(Y_ood, axis = (1,2)))
#print(f'Gaussian kernel: test error is {e:.2e}.\n')    

Matern kernel: test error is 4.90e-04.



# Vanilla Kernel

In [16]:
num = int(0.8 * dataset.shape[0])
X_train = dataset[:num,0,:,0]
Y_train = dataset[:num,:,:,0]
X_test = dataset[num:,0,:,0]
Y_test = dataset[num:,:,:,0]
coef_train = coef[:num]
coef_test = coef[num:]
print(X_train.shape, coef_train.shape, Y_train.shape)

# out-of-distribution
#X_ood = dataset_ood[:,0,:,0]
#Y_ood = dataset_ood[:,:,:,0]
#coef_ood = coef_ood
#print('Out-of-distribution shapes:', X_ood.shape, Y_ood.shape)

##################### RBF kernel
bandwidth = 100
kernel = RBF(length_scale = bandwidth)
model = GaussianProcessRegressor(kernel, alpha = 1e-10, optimizer=None)
model.fit(X_train, Y_train.reshape(num,64*128))

# prediction
pred = model.predict(X_test)
pred = pred.reshape(Y_test.shape)
e = np.mean(np.linalg.norm(pred - Y_test, axis = (1,2))/np.linalg.norm(Y_test, axis = (1,2)))
print(f'Test error is {e:.2e}.\n')

(6400, 128) (6400, 129) (6400, 64, 128)
Test error is 6.78e-02.



# Framework 2

In [18]:
# kernel for coefficients
bandwidth1 = 1
nu1 = 5/2
kernel1 = Matern(length_scale = bandwidth1, nu=nu1)
K_c = kernel1(coef_train)

# kernel for u
bandwidth2 = 0.1
nu2 = 5/2
kernel2 = Matern(length_scale = bandwidth2, nu=nu2)
K_u = kernel2(X_train)

# product kernel
K_train = K_c * K_u

# model
model = KernelRidge(alpha = 1e-10, kernel='precomputed')
model.fit(K_train, Y_train.reshape(num,64*128))

# prediction
K_c_test = kernel1(coef_test, coef_train)
K_u_test = kernel2(X_test, X_train)
K_test = K_c_test * K_u_test
our_pred = model.predict(K_test)
our_pred = our_pred.reshape(Y_test.shape)

# Error
our_e = np.mean(np.linalg.norm(our_pred - Y_test, axis = (1,2))/np.linalg.norm(Y_test, axis = (1,2)))
print(f'Test error of our method is {our_e:.2e}.\n')

Test error of our method is 6.45e-04.

